# Deep Behavior Analysis: Understanding the Current 2,700 Runs

This notebook explains the deeper analysis requested by Professor Ahmed.

The goal is not only to ask whether neutral prompts perform better, but **why** they perform better.

Main question:

> Does the tone effect remain after controlling for output length, code-only response rate, parsing/extraction failures, and runtime?

If the tone effect becomes much smaller after adding these controls, then the result is better interpreted as:

> Prompt tone changes output behavior, and that behavior affects downstream coding performance.

## 1. Setup

We load the same three result files used in the previous report:

- `gemma2-9b_final.jsonl`
- `llama3.1-8b_final.jsonl`
- `qwen2.5-7b_final.jsonl`

We also load:

- `problems.csv`
- `prompt_matrix_prompt_template.08.csv`

The result `task_id` has the format `problemID_templateID`, for example `1_7`.

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
PROBLEMS_PATH = ROOT / "problems.csv"
PROMPT_MATRIX_PATH = ROOT / "prompt_matrix_prompt_template.08.csv"
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = ROOT / "analysis_outputs" / "deep_behavior_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", ROOT)
print("Output directory:", OUTPUT_DIR)

## 2. Load and Join Data

This cell reads all JSONL result files and joins them with prompt-template metadata.

Important joined columns:

- `model_short`: model name from the result file
- `problem_id`: coding problem id
- `template_id`: prompt template id
- `Tone`, `Role_framing`, `Format`: prompt factors
- `passed`, `tests_passed`, `tests_total`: evaluation result
- `completion_tokens`, `duration_seconds`: behavior/runtime metrics
- `response_text`, `extracted_code`: raw output and extracted code

In [ ]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["source_file"] = path.name
            rows.append(row)
    return pd.DataFrame(rows)


def parse_result_task_id(value: object) -> tuple[int, int]:
    match = re.fullmatch(r"(\d+)_(\d+)", str(value))
    if not match:
        raise ValueError(f"Unexpected result task_id format: {value!r}")
    return int(match.group(1)), int(match.group(2))


def short_model_name(source_file: str, model_value: object = None) -> str:
    if source_file.endswith("_final.jsonl"):
        return source_file.removesuffix("_final.jsonl")
    if model_value is not None:
        return str(model_value).replace(":", "-")
    return source_file.removesuffix(".jsonl")


problems = pd.read_csv(PROBLEMS_PATH)
prompt_matrix = pd.read_csv(PROMPT_MATRIX_PATH).rename(columns={"ID": "template_id", "Role framing": "Role_framing"})
result_paths = sorted(RESULTS_DIR.glob("*_final.jsonl"))

results = pd.concat([read_jsonl(path) for path in result_paths], ignore_index=True)
parsed_ids = results["task_id"].map(parse_result_task_id)
results["problem_id"] = parsed_ids.map(lambda pair: pair[0])
results["template_id"] = parsed_ids.map(lambda pair: pair[1])
results["model_short"] = [
    short_model_name(src, model)
    for src, model in zip(results["source_file"], results.get("model", pd.Series([None] * len(results))))
]

problem_meta = (
    problems.rename(columns={"task_id": "problem_id"})
    .sort_values(["problem_id", "category_id"])
    .drop_duplicates("problem_id")
    [["problem_id", "prompt", "test_cases", "challenge_test_list"]]
)

df = (
    results.merge(prompt_matrix, on="template_id", how="left", validate="many_to_one")
    .merge(problem_meta, on="problem_id", how="left", validate="many_to_one", suffixes=("", "_problem"))
)

print("Result files:", [p.name for p in result_paths])
print("Rows:", len(df))
print("Models:", df["model_short"].unique())
print("Problems:", df["problem_id"].nunique())
print("Templates:", df["template_id"].nunique())
df.head(3)

## 3. Create Derived Metrics

We create the variables used in the deep analysis.

Performance metrics:

- `passed_int`: exact pass/fail as 0/1
- `tests_fraction`: fraction of unit tests passed

Behavior metrics:

- `completion_tokens`: output length
- `duration_seconds`: runtime/latency
- `likely_code_only`: whether the response is mostly extracted code
- `extraction_failed`: whether no code was extracted
- `runtime_failure`: syntax/name/import/timeout/empty-code failure proxy
- `extra_text_ratio`: share of response text outside extracted code

Important caveat: `extraction_failed` is a conservative proxy for parsing failure. It catches empty extracted code, but it is not a complete parser-failure taxonomy.

In [ ]:
def safe_len(value: object) -> int:
    if isinstance(value, str):
        return len(value)
    if pd.isna(value):
        return 0
    return len(str(value))


def count_lines(value: object) -> int:
    if not isinstance(value, str) or not value.strip():
        return 0
    return len(value.splitlines())


def error_text(errors: object) -> str:
    if isinstance(errors, list):
        return "\n".join(map(str, errors))
    if pd.isna(errors):
        return ""
    return str(errors)


def classify_error(errors: object, extracted_code: object) -> str:
    text = error_text(errors).lower()
    code_value = extracted_code if isinstance(extracted_code, str) else ""
    if not code_value.strip():
        return "missing_or_empty_code"
    if not text.strip():
        return "none"
    if "not defined" in text:
        return "name_not_defined"
    if "syntax" in text or "invalid syntax" in text:
        return "syntax_error"
    if "timeout" in text or "timed out" in text:
        return "timeout"
    if "importerror" in text or "no module named" in text:
        return "import_error"
    if "assert" in text or "test failed" in text:
        return "wrong_answer"
    return "other_runtime"


df["passed_int"] = df["passed"].astype(bool).astype(int)
df["tests_passed"] = pd.to_numeric(df["tests_passed"], errors="coerce")
df["tests_total"] = pd.to_numeric(df["tests_total"], errors="coerce")
df["tests_fraction"] = np.where(df["tests_total"] > 0, df["tests_passed"] / df["tests_total"], np.nan)
df["prompt_tokens"] = pd.to_numeric(df["prompt_tokens"], errors="coerce")
df["completion_tokens"] = pd.to_numeric(df["completion_tokens"], errors="coerce")
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")

df["response_chars"] = df["response_text"].map(safe_len)
df["code_chars"] = df["extracted_code"].map(safe_len)
df["response_lines"] = df["response_text"].map(count_lines)
df["code_lines"] = df["extracted_code"].map(count_lines)
df["has_extracted_code"] = (df["code_chars"] > 0).astype(int)
df["extraction_failed"] = (df["has_extracted_code"] == 0).astype(int)
df["extra_text_chars"] = (df["response_chars"] - df["code_chars"]).clip(lower=0)
df["extra_text_ratio"] = np.where(df["response_chars"] > 0, df["extra_text_chars"] / df["response_chars"], np.nan)
df["likely_code_only"] = ((df["has_extracted_code"] == 1) & (df["extra_text_ratio"] < 0.15)).astype(int)
df["error_text"] = df["errors"].map(error_text)
df["error_category"] = [classify_error(e, c) for e, c in zip(df["errors"], df["extracted_code"])]
df["runtime_failure"] = df["error_category"].isin([
    "syntax_error", "name_not_defined", "import_error", "other_runtime", "timeout", "missing_or_empty_code"
]).astype(int)

df["log_completion_tokens"] = np.log1p(df["completion_tokens"])
df["log_duration_seconds"] = np.log1p(df["duration_seconds"])

cols = [
    "model_short", "problem_id", "template_id", "Tone", "Role_framing", "Format",
    "passed_int", "tests_fraction", "completion_tokens", "duration_seconds",
    "likely_code_only", "extraction_failed", "runtime_failure", "extra_text_ratio", "error_category",
]
df[cols].head()

## 4. Descriptive Summary by Tone

This is the first important table.

It shows whether tone changes both performance and output behavior.

Look especially at:

- `completion_tokens`
- `duration_seconds`
- `likely_code_only`
- `passed_int`
- `tests_fraction`

If neutral prompts are shorter, faster, and more code-only, then tone is changing behavior, not only correctness.

In [ ]:
tone_summary = (
    df.groupby("Tone")
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        tests_fraction=("tests_fraction", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        extraction_failure_rate=("extraction_failed", "mean"),
        runtime_failure_rate=("runtime_failure", "mean"),
        extra_text_ratio=("extra_text_ratio", "mean"),
    )
    .reset_index()
)

tone_summary.to_csv(OUTPUT_DIR / "tone_summary.csv", index=False)
tone_summary.style.format({
    "pass_rate": "{:.1%}",
    "tests_fraction": "{:.1%}",
    "avg_completion_tokens": "{:.0f}",
    "avg_duration_seconds": "{:.2f}",
    "code_only_rate": "{:.1%}",
    "extraction_failure_rate": "{:.1%}",
    "runtime_failure_rate": "{:.1%}",
    "extra_text_ratio": "{:.1%}",
})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_specs = [
    ("tests_fraction", "Average test fraction"),
    ("completion_tokens", "Completion tokens"),
    ("duration_seconds", "Duration seconds"),
    ("likely_code_only", "Code-only rate"),
]
for ax, (metric, title) in zip(axes.flat, plot_specs):
    sns.barplot(data=df, x="Tone", y=metric, errorbar=("ci", 95), ax=ax, order=["neutral", "polite", "aggressive"])
    ax.set_title(title)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

### Interpretation

The descriptive result should show:

- neutral prompts have the best average performance
- neutral prompts are much shorter
- neutral prompts are much more likely to be code-only
- polite/aggressive prompts are slower and more verbose

This motivates the next step: regression with behavior controls.

## 5. Fixed-Effect Regression: Baseline Versus Behavior Controls

This is the core analysis.

We estimate models like:

```text
outcome ~ Tone + Role_framing + Format + model fixed effects + problem fixed effects + behavior controls
```

Why fixed effects?

- `C(model_short)` controls for different baseline model ability.
- `C(problem_id)` controls for different problem difficulty.

The key coefficient is:

```text
C(Tone)[T.neutral]
```

The hidden reference category is aggressive tone. So the neutral coefficient means:

> How much better/worse neutral is compared with aggressive, after controlling for the other variables.

In [ ]:
def coef_rows(result, model_name: str, outcome: str) -> pd.DataFrame:
    rows = result.params.rename("coef").to_frame()
    rows["std_err"] = result.bse
    rows["p_value"] = result.pvalues
    rows["model_spec"] = model_name
    rows["outcome"] = outcome
    rows["term"] = rows.index
    return rows.reset_index(drop=True)


def fit_ols(data: pd.DataFrame, outcome: str, controls: list[str], spec_name: str):
    prompt_terms = "C(Tone) + C(Role_framing) + C(Format)"
    fixed_effects = "C(model_short) + C(problem_id)"
    formula = f"{outcome} ~ " + " + ".join([prompt_terms, fixed_effects, *controls])
    model_cols = [outcome, "Tone", "Role_framing", "Format", "model_short", "problem_id", *controls]
    model_data = data[model_cols].dropna()
    result = smf.ols(formula, data=model_data).fit(cov_type="HC3")
    return result, formula


control_sets = {
    "M0_prompt_only": [],
    "M1_length": ["log_completion_tokens"],
    "M2_code_only": ["likely_code_only"],
    "M3_extraction": ["extraction_failed"],
    "M4_runtime": ["log_duration_seconds"],
    "M5_all_behavior_controls": ["log_completion_tokens", "likely_code_only", "extraction_failed", "log_duration_seconds"],
}

ols_rows = []
formulas = []
for outcome in ["passed_int", "tests_fraction"]:
    for spec_name, controls in control_sets.items():
        result, formula = fit_ols(df, outcome, controls, spec_name)
        ols_rows.append(coef_rows(result, spec_name, outcome))
        formulas.append({"outcome": outcome, "model_spec": spec_name, "formula": formula, "nobs": result.nobs, "rsquared": result.rsquared})

ols_results = pd.concat(ols_rows, ignore_index=True)
ols_formulas = pd.DataFrame(formulas)
ols_results.to_csv(OUTPUT_DIR / "behavior_control_ols_models.csv", index=False)
ols_formulas.to_csv(OUTPUT_DIR / "behavior_control_ols_formulas.csv", index=False)

focus_terms = [
    "C(Tone)[T.neutral]", "C(Tone)[T.polite]",
    "log_completion_tokens", "likely_code_only", "extraction_failed", "log_duration_seconds",
]
focus_ols = ols_results[ols_results["term"].isin(focus_terms)].copy()
focus_ols[["outcome", "model_spec", "term", "coef", "std_err", "p_value"]].style.format({
    "coef": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}"
})

## 6. Read the Regression Result

Focus on `C(Tone)[T.neutral]`.

Expected pattern from the current run:

- In `M0_prompt_only`, neutral is significantly better.
- After adding output length, the neutral effect becomes small/non-significant.
- After adding code-only compliance, the neutral effect becomes small/non-significant.
- After adding runtime, the neutral effect becomes small/non-significant.
- With all behavior controls, the neutral effect is no longer significant.

This supports the interpretation:

> The neutral-tone advantage is largely explained by output behavior.

Important nuance:

`extraction_failed` alone does not explain the neutral effect. This is because neutral prompts actually had more empty extracted-code cases than polite prompts, mostly from Llama. So extraction failure is important, but it is not the main explanation of neutral's advantage.

In [ ]:
neutral_summary = (
    focus_ols[focus_ols["term"].eq("C(Tone)[T.neutral]")]
    .pivot_table(index=["outcome", "model_spec"], values=["coef", "p_value"], aggfunc="first")
    .reset_index()
)
neutral_summary.to_csv(OUTPUT_DIR / "neutral_effect_by_control_model.csv", index=False)
neutral_summary.style.format({"coef": "{:+.4f}", "p_value": "{:.4f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_data = focus_ols[(focus_ols["term"] == "C(Tone)[T.neutral]") & (focus_ols["outcome"] == "tests_fraction")].copy()
plot_data["ci_low"] = plot_data["coef"] - 1.96 * plot_data["std_err"]
plot_data["ci_high"] = plot_data["coef"] + 1.96 * plot_data["std_err"]

ax.errorbar(
    x=plot_data["coef"],
    y=plot_data["model_spec"],
    xerr=[plot_data["coef"] - plot_data["ci_low"], plot_data["ci_high"] - plot_data["coef"]],
    fmt="o",
    capsize=4,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Neutral-tone coefficient for tests_fraction under behavior controls")
ax.set_xlabel("Coefficient vs aggressive tone")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 7. Logistic Robustness Check

Because `passed_int` is binary, OLS is easy to interpret but not the only option.

This cell runs a binomial GLM/logit model for pass/fail.

Question:

> Does the same qualitative pattern appear when using a binary-outcome model?

In the current data, yes: the baseline neutral effect is significant, but after behavior controls it is no longer significant.

In [ ]:
def fit_glm_binomial(data: pd.DataFrame, controls: list[str], spec_name: str):
    prompt_terms = "C(Tone) + C(Role_framing) + C(Format)"
    fixed_effects = "C(model_short) + C(problem_id)"
    formula = "passed_int ~ " + " + ".join([prompt_terms, fixed_effects, *controls])
    model_cols = ["passed_int", "Tone", "Role_framing", "Format", "model_short", "problem_id", *controls]
    model_data = data[model_cols].dropna()
    result = smf.glm(formula=formula, data=model_data, family=sm.families.Binomial()).fit(cov_type="HC3")
    return result, formula


glm_rows = []
glm_formulas = []
for spec_name, controls in control_sets.items():
    result, formula = fit_glm_binomial(df, controls, spec_name)
    glm_rows.append(coef_rows(result, spec_name, "passed_int_glm_binomial_logit"))
    glm_formulas.append({"outcome": "passed_int", "model_spec": spec_name, "formula": formula, "nobs": result.nobs, "aic": result.aic})

glm_results = pd.concat(glm_rows, ignore_index=True)
glm_results.to_csv(OUTPUT_DIR / "behavior_control_glm_logit_models.csv", index=False)

focus_glm = glm_results[glm_results["term"].isin(focus_terms)].copy()
focus_glm[["model_spec", "term", "coef", "std_err", "p_value"]].style.format({
    "coef": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}"
})

## 8. Behavior Metric Correlations

This table checks how behavior variables relate to each other.

Important result from the current data:

- `completion_tokens` and `duration_seconds` are almost perfectly correlated.

That means when both are included together, their individual coefficients are hard to interpret. The all-controls model is useful as a robustness check, but not as a clean separation of length versus runtime.

In [ ]:
corr_cols = [
    "passed_int", "tests_fraction", "completion_tokens", "duration_seconds",
    "likely_code_only", "extraction_failed", "runtime_failure", "extra_text_ratio",
]
correlations = df[corr_cols].corr(method="pearson")
correlations.to_csv(OUTPUT_DIR / "behavior_metric_correlations.csv")
correlations.style.format("{:.3f}").background_gradient(cmap="coolwarm", vmin=-1, vmax=1)

## 9. Model-Specific Analysis

Now we check whether the tone effect is consistent across models.

This matters because a publishable paper should avoid overclaiming.

Question:

> Is neutral tone best for every model, or mainly best overall?

In [ ]:
model_tone = (
    df.groupby(["model_short", "Tone"])
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        tests_fraction=("tests_fraction", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        extraction_failure_rate=("extraction_failed", "mean"),
        runtime_failure_rate=("runtime_failure", "mean"),
    )
    .reset_index()
    .sort_values(["model_short", "tests_fraction"], ascending=[True, False])
)
model_tone.to_csv(OUTPUT_DIR / "model_by_tone_behavior_summary.csv", index=False)
model_tone.style.format({
    "pass_rate": "{:.1%}",
    "tests_fraction": "{:.1%}",
    "avg_completion_tokens": "{:.0f}",
    "avg_duration_seconds": "{:.2f}",
    "code_only_rate": "{:.1%}",
    "extraction_failure_rate": "{:.1%}",
    "runtime_failure_rate": "{:.1%}",
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=df, x="model_short", y="tests_fraction", hue="Tone", errorbar=("ci", 95), ax=axes[0], hue_order=["neutral", "polite", "aggressive"])
axes[0].set_title("Test fraction by model and tone")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=df, x="model_short", y="likely_code_only", hue="Tone", errorbar=("ci", 95), ax=axes[1], hue_order=["neutral", "polite", "aggressive"])
axes[1].set_title("Code-only rate by model and tone")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### Interpretation

In the current result:

- neutral is strongest for Gemma and Qwen
- Llama is the caveat: polite is slightly higher by `tests_fraction`
- neutral has high extraction failure for Llama compared with the other models

So the safe wording is:

> Neutral tone produced the strongest overall result, but the effect is mediated by output behavior and varies across model families.

## 10. Difficulty Analysis

A prompt technique may help on medium/easy tasks but not on tasks that are too hard for all models.

Here we split problems into three difficulty bands based on average `tests_fraction` across all runs.

In [ ]:
problem_difficulty = (
    df.groupby("problem_id")
    .agg(problem_mean_tests_fraction=("tests_fraction", "mean"), problem_pass_rate=("passed_int", "mean"))
    .reset_index()
)
problem_difficulty["difficulty_band"] = pd.qcut(
    problem_difficulty["problem_mean_tests_fraction"],
    q=3,
    labels=["hard", "medium", "easy"],
    duplicates="drop",
)

df = df.drop(columns=["difficulty_band"], errors="ignore").merge(
    problem_difficulty[["problem_id", "difficulty_band"]], on="problem_id", how="left"
)

difficulty_tone = (
    df.groupby(["difficulty_band", "Tone"], observed=False)
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        tests_fraction=("tests_fraction", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        extraction_failure_rate=("extraction_failed", "mean"),
        runtime_failure_rate=("runtime_failure", "mean"),
    )
    .reset_index()
)
difficulty_tone.to_csv(OUTPUT_DIR / "difficulty_by_tone_behavior_summary.csv", index=False)
difficulty_tone.style.format({
    "pass_rate": "{:.1%}",
    "tests_fraction": "{:.1%}",
    "avg_completion_tokens": "{:.0f}",
    "avg_duration_seconds": "{:.2f}",
    "code_only_rate": "{:.1%}",
    "extraction_failure_rate": "{:.1%}",
    "runtime_failure_rate": "{:.1%}",
})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=df, x="difficulty_band", y="tests_fraction", hue="Tone", errorbar=("ci", 95), ax=ax, hue_order=["neutral", "polite", "aggressive"])
ax.set_title("Tone effect by problem difficulty")
ax.set_xlabel("Difficulty band")
ax.set_ylabel("Average test fraction")
plt.tight_layout()
plt.show()

### Interpretation

The current data suggests:

- hard problems are near zero for all tones
- neutral helps most on medium and easy problems
- prompt tone cannot rescue problems that are beyond the current model/problem setup

## 11. Error Mode Analysis

This checks what kind of failures happen under each tone.

The most useful categories are:

- `wrong_answer`
- `name_not_defined`
- `missing_or_empty_code`
- `timeout`
- `syntax_error`

In [ ]:
error_summary = (
    df.groupby(["Tone", "error_category"])
    .size()
    .rename("runs")
    .reset_index()
)
error_summary["rate"] = error_summary["runs"] / error_summary.groupby("Tone")["runs"].transform("sum")
error_summary = error_summary.sort_values(["Tone", "runs"], ascending=[True, False])
error_summary.to_csv(OUTPUT_DIR / "tone_error_category_summary.csv", index=False)
error_summary.style.format({"rate": "{:.1%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_errors = error_summary[error_summary["error_category"].ne("none")].copy()
sns.barplot(data=plot_errors, x="error_category", y="rate", hue="Tone", ax=ax, hue_order=["neutral", "polite", "aggressive"])
ax.set_title("Failure category rate by tone")
ax.set_xlabel("")
ax.set_ylabel("Rate within tone")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 12. Final Interpretation

The strongest current interpretation is:

> Tone affects downstream coding performance mostly by changing output behavior. Neutral prompts lead to shorter, faster, more code-only responses, which fit the evaluator better. Polite and aggressive prompts encourage longer explanatory responses, which increase latency and reduce code-only compliance; these behavioral shifts explain much of the observed performance gap.

This is stronger than saying only:

> Neutral prompts are better.

because it explains a plausible mechanism.

## Caveats to Mention

- Behavior controls are post-treatment variables. They help explain the mechanism, but they are not pure causal controls.
- Completion tokens and runtime are highly correlated, so do not overinterpret them separately in the all-controls model.
- `extraction_failed` is a conservative proxy for parsing failure.
- The effect is not universal across every model; Llama is the main exception.
- Exact pass rates are low, so `tests_fraction` is often more informative.

## Suggested Paper Framing

Use this framing:

> a controlled study of how prompt tone changes output behavior, compliance, latency, verbosity, and downstream task performance.

Avoid this weaker framing:

> testing many prompts on many models.